## 0. Define the Prediction Task 

- Our prediction target in this ML experience will be the Next Day Return "log_return_1d" shifted by 1 day . This feature predicts tomorrow's % change . why not predict the price directly ? 


- Scale Differences: Coins prices vary widely (some trade at $1, others at $100,000).
Log returns have the same scale, ex:a +2% move in Bitcoin and a +2% move in Ethereum are treated equivalently by the model.

- Easier Learning Task for the Model :Predicting percentage change is easier for the model to learn than the raw price.

- More Relevant for traders: Traders care more about returns than raw prices.What matters is whether the asset will go up or down, and by how much in percentage 


## 1. Load Libraries , data and create target variable

In [1]:

# Load libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


In [2]:
# ============================================
# LOAD DATA AND CREATE TARGET VARIABLE
# ============================================

df = pd.read_csv(
    r"C:\Users\tlili\OneDrive\Bureau\Bootcamp\AI-powered-cryptocurrency-market-analysis-and-decision-support-system\data\processed\crypto_clean_2026-02-12.csv",
    parse_dates=["timestamp"]
)

# Sort by coin and timestamp
df = df.sort_values(["coin", "timestamp"]).reset_index(drop=True)

# Target = next day's log return
df["target"] = df.groupby("coin")["log_return_1d"].shift(-1)

# Drop last row per coin (no next day available)
rows_before = len(df)
df = df.dropna(subset=["target"]).reset_index(drop=True)
rows_after = len(df)

print(f"✅ Target variable created!")
print(f"   Rows before: {rows_before}")
print(f"   Rows after:  {rows_after}")
print(f"   Rows dropped: {rows_before - rows_after} (last day per coin)")
print(f"\nTarget statistics:")
print(df["target"].describe())

✅ Target variable created!
   Rows before: 6346
   Rows after:  6344
   Rows dropped: 2 (last day per coin)

Target statistics:
count    6344.000000
mean        0.000947
std         0.040924
min        -0.550714
25%        -0.016196
50%         0.000890
75%         0.018985
max         0.234731
Name: target, dtype: float64


insights for the return distribution: ( goal is to understand if the target is really scaled and if it has weired outliers that could break the model )

- Mean-0.0008≈ 0% → Market neutral over the year 
- Std0.0335 = 3.35% daily volatility → Typical for crypto
- Min-0.167  -16.7% worst drop → Feb crash!
- Max+0.194  +19.4% best pump → Recovery bounce!
- Median-0.00013 Almost exactly 0 → Symmetric

--> the returns are scaled that means we can train 1 model for both coins 

In [4]:
# ============================================
#  CREATE LAGGED FEATURES (NO DATA LEAKAGE)
# ============================================

print("="*60)
print("CREATING LAGGED FEATURES")
print("="*60)

# All technical features need to be lagged by 1 day
# Reason: On day T, we only know day T-1's indicators
# We use T-1 features to predict T+1 return

features_to_lag = [
    # Raw market data
    "volume",
    # Volatility & momentum indicators
    "volatility_7d", "volatility_14d", "rsi_14",
    # Moving averages
    "ma_7", "ma_14", "ma_30",
    "price_to_ma7", "price_to_ma30",
    # MACD
    "macd", "macd_signal", "macd_histogram",
    # Momentum
    "momentum_5d", "momentum_10d", "momentum_acceleration"
]

# Create lagged features per coin
for feature in features_to_lag:
    df[f"{feature}_lag1"] = df.groupby("coin")[feature].shift(1)

print(f"✅ Created {len(features_to_lag)} lagged features!")

# Drop NaNs from lagging (first row per coin)
rows_before = len(df)
lag_cols = [f"{f}_lag1" for f in features_to_lag]
df = df.dropna(subset=lag_cols).reset_index(drop=True)
rows_after = len(df)

print(f"   Rows before: {rows_before}")
print(f"   Rows after:  {rows_after}")
print(f"   Rows dropped: {rows_before - rows_after}")

# Verify lag is correct
print(f"\n🔍 Lag verification (Bitcoin, first 4 rows):")
btc = df[df["coin"] == "bitcoin"][
    ["timestamp", "rsi_14", "rsi_14_lag1"]
].head(4)
print(btc)
print("\n✅ rsi_14_lag1 should be one day behind rsi_14!")

CREATING LAGGED FEATURES
✅ Created 15 lagged features!
   Rows before: 6344
   Rows after:  6342
   Rows dropped: 2

🔍 Lag verification (Bitcoin, first 4 rows):
Empty DataFrame
Columns: [timestamp, rsi_14, rsi_14_lag1]
Index: []

✅ rsi_14_lag1 should be one day behind rsi_14!


## 2. Feature selection and data split 

In [6]:
# ============================================
# BUILD FEATURE MATRIX
# ============================================

print("="*60)
print("FEATURE MATRIX")
print("="*60)

# Use ONLY lagged features (no leakage!)
lagged_cols = [f"{f}_lag1" for f in features_to_lag]


# Coin dummies (one-hot encoded)
coin_dummies = pd.get_dummies(df["coin"], prefix="coin").astype(int)

# Build X and y
X = df[lagged_cols].copy()
X = pd.concat([X, coin_dummies], axis=1)
y = df["target"].copy()

print(f"✅ Feature matrix created!")
print(f"   Shape: {X.shape}")
print(f"   Features: {X.shape[1]}")
print(f"   Samples: {X.shape[0]}")

print(f"\n🔍 Data quality check:")
print(f"   NaNs in X: {X.isna().sum().sum()}")
print(f"   NaNs in y: {y.isna().sum()}")


print(f"\nFeature list:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

FEATURE MATRIX
✅ Feature matrix created!
   Shape: (6342, 17)
   Features: 17
   Samples: 6342

🔍 Data quality check:
   NaNs in X: 0
   NaNs in y: 0

Feature list:
   1. volume_lag1
   2. volatility_7d_lag1
   3. volatility_14d_lag1
   4. rsi_14_lag1
   5. ma_7_lag1
   6. ma_14_lag1
   7. ma_30_lag1
   8. price_to_ma7_lag1
   9. price_to_ma30_lag1
  10. macd_lag1
  11. macd_signal_lag1
  12. macd_histogram_lag1
  13. momentum_5d_lag1
  14. momentum_10d_lag1
  15. momentum_acceleration_lag1
  16. coin_btc
  17. coin_eth


In [7]:
# ============================================
# TRAIN/VAL/TEST SPLIT (TIME-AWARE, PER COIN)
# ============================================

print("="*60)
print("TRAIN/VALIDATION/TEST SPLIT")
print("="*60)

train_ratio = 0.60
val_ratio   = 0.20
test_ratio  = 0.20

train_X_list, val_X_list, test_X_list = [], [], []
train_y_list, val_y_list, test_y_list = [], [], []
train_df_list, val_df_list, test_df_list = [], [], []

for coin_name in df["coin"].unique():

    # Filter per coin
    coin_mask = df["coin"] == coin_name
    coin_df   = df[coin_mask].copy()
    coin_X    = X[coin_mask].copy()
    coin_y    = y[coin_mask].copy()

    # Calculate split indices
    n         = len(coin_df)
    train_idx = int(n * train_ratio)
    val_idx   = int(n * (train_ratio + val_ratio))

    print(f"\n📊 {coin_name.upper()}:")
    print(f"   Total: {n} samples")
    print(f"   Train: {train_idx} samples "
          f"({coin_df.iloc[0]['timestamp'].date()} → "
          f"{coin_df.iloc[train_idx-1]['timestamp'].date()})")
    print(f"   Val:   {val_idx - train_idx} samples "
          f"({coin_df.iloc[train_idx]['timestamp'].date()} → "
          f"{coin_df.iloc[val_idx-1]['timestamp'].date()})")
    print(f"   Test:  {n - val_idx} samples "
          f"({coin_df.iloc[val_idx]['timestamp'].date()} → "
          f"{coin_df.iloc[-1]['timestamp'].date()})")

    # Append splits
    train_X_list.append(coin_X.iloc[:train_idx])
    val_X_list.append(coin_X.iloc[train_idx:val_idx])
    test_X_list.append(coin_X.iloc[val_idx:])

    train_y_list.append(coin_y.iloc[:train_idx])
    val_y_list.append(coin_y.iloc[train_idx:val_idx])
    test_y_list.append(coin_y.iloc[val_idx:])

    train_df_list.append(coin_df.iloc[:train_idx])
    val_df_list.append(coin_df.iloc[train_idx:val_idx])
    test_df_list.append(coin_df.iloc[val_idx:])

# Combine all coins
X_train = pd.concat(train_X_list).reset_index(drop=True)
X_val   = pd.concat(val_X_list).reset_index(drop=True)
X_test  = pd.concat(test_X_list).reset_index(drop=True)

y_train = pd.concat(train_y_list).reset_index(drop=True)
y_val   = pd.concat(val_y_list).reset_index(drop=True)
y_test  = pd.concat(test_y_list).reset_index(drop=True)

train_df = pd.concat(train_df_list).reset_index(drop=True)
val_df   = pd.concat(val_df_list).reset_index(drop=True)
test_df  = pd.concat(test_df_list).reset_index(drop=True)

# Summary
print(f"\n{'='*60}")
print(f"✅ FINAL SPLIT SUMMARY:")
print(f"   Train: {len(X_train)} samples")
print(f"   Val:   {len(X_val)} samples")
print(f"   Test:  {len(X_test)} samples")
print(f"   Total: {len(X_train)+len(X_val)+len(X_test)} ✓")#

# Verify no overlap
print(f"\n🔍 Verification (per coin):")
for coin_name in df["coin"].unique():
    tr_max = train_df[train_df["coin"]==coin_name]["timestamp"].max()
    v_min  = val_df[val_df["coin"]==coin_name]["timestamp"].min()
    v_max  = val_df[val_df["coin"]==coin_name]["timestamp"].max()
    te_min = test_df[test_df["coin"]==coin_name]["timestamp"].min()

    print(f"\n   {coin_name.upper()}:")
    print(f"   Train ends:  {tr_max.date()}")
    print(f"   Val starts:  {v_min.date()} "
          f"{'✅' if v_min > tr_max else '❌'}")
    print(f"   Val ends:    {v_max.date()}")
    print(f"   Test starts: {te_min.date()} "
          f"{'✅' if te_min > v_max else '❌'}")#

TRAIN/VALIDATION/TEST SPLIT

📊 BTC:
   Total: 3327 samples
   Train: 1996 samples (2017-01-02 → 2022-06-20)
   Val:   665 samples (2022-06-21 → 2024-04-15)
   Test:  666 samples (2024-04-16 → 2026-02-10)

📊 ETH:
   Total: 3015 samples
   Train: 1809 samples (2017-11-10 → 2022-10-23)
   Val:   603 samples (2022-10-24 → 2024-06-17)
   Test:  603 samples (2024-06-18 → 2026-02-10)

✅ FINAL SPLIT SUMMARY:
   Train: 3805 samples
   Val:   1268 samples
   Test:  1269 samples
   Total: 6342 ✓

🔍 Verification (per coin):

   BTC:
   Train ends:  2022-06-20
   Val starts:  2022-06-21 ✅
   Val ends:    2024-04-15
   Test starts: 2024-04-16 ✅

   ETH:
   Train ends:  2022-10-23
   Val starts:  2022-10-24 ✅
   Val ends:    2024-06-17
   Test starts: 2024-06-18 ✅


- Train 436 (218×2 coins) Feb 12 → Sep 17 2025
- Validation 146 (73×2 coins)Sep 18 → Nov 29 2025
- Test 146 (73×2 coins)Nov 30 → Feb 10 2026
- Total 728

## 3. Features scaling 

- Before training, we need to scale our features. Here's why:market_cap:1,900,000,000,000(trillions)
, rsi_14:45.5(0-100) and has_max_supply: 1 (binary)
- Model might think market_cap is more important just because its a bigger number

In [8]:
# ============================================
#  FEATURE SCALING
# ============================================

from sklearn.preprocessing import StandardScaler

print("="*60)
print("FEATURE SCALING")
print("="*60)

# Binary columns - no scaling needed
no_scale_cols = ["coin_bitcoin", "coin_ethereum"]

# Columns to scale
scale_cols = [col for col in X_train.columns if col not in no_scale_cols]

print(f"\n📊 Columns to scale:    {len(scale_cols)}")
print(f"📊 Columns unchanged:   {len(no_scale_cols)}")

# Fit scaler on TRAIN ONLY (no data leakage!)
scaler = StandardScaler()
scaler.fit(X_train[scale_cols])

# Transform all sets using same scaler
X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[scale_cols] = scaler.transform(X_train[scale_cols])
X_val_scaled[scale_cols]   = scaler.transform(X_val[scale_cols])
X_test_scaled[scale_cols]  = scaler.transform(X_test[scale_cols])

# Verify
print(f"\n🔍 Verification (mean≈0, std≈1 for train):")
stats = X_train_scaled[scale_cols].describe().loc[["mean","std"]].round(3)
print(stats)

print(f"\n⚠️  Scaler fitted on TRAIN only - no data leakage! ✅")
print("="*60)


FEATURE SCALING

📊 Columns to scale:    17
📊 Columns unchanged:   2

🔍 Verification (mean≈0, std≈1 for train):
      volume_lag1  volatility_7d_lag1  volatility_14d_lag1  rsi_14_lag1  \
mean          0.0                -0.0                 -0.0          0.0   
std           1.0                 1.0                  1.0          1.0   

      ma_7_lag1  ma_14_lag1  ma_30_lag1  price_to_ma7_lag1  \
mean        0.0        -0.0         0.0               -0.0   
std         1.0         1.0         1.0                1.0   

      price_to_ma30_lag1  macd_lag1  macd_signal_lag1  macd_histogram_lag1  \
mean                -0.0        0.0              -0.0                 -0.0   
std                  1.0        1.0               1.0                  1.0   

      momentum_5d_lag1  momentum_10d_lag1  momentum_acceleration_lag1  \
mean              -0.0                0.0                        -0.0   
std                1.0                1.0                         1.0   

      coin_btc  coin_

## 4. Machine learning models  

#### 4.1 XGBoost 

In [9]:
# ============================================
#  EVALUATION FUNCTION
# ============================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_model(y_true, y_pred, set_name=""):
    """
    Evaluate model performance.
    Returns MAE, RMSE, R², and Directional Accuracy.
    """
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    # Directional accuracy (most important for trading!)
    dir_acc = (np.sign(y_true) == np.sign(y_pred)).mean()

    print(f"\n📊 {set_name} Performance:")
    print(f"   MAE:                  {mae:.6f}  ({mae*100:.4f}%)")
    print(f"   RMSE:                 {rmse:.6f}  ({rmse*100:.4f}%)")
    print(f"   R²:                   {r2:.4f}")
    print(f"   Directional Accuracy: {dir_acc:.4f} ({dir_acc*100:.2f}%)")

    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "directional_accuracy": dir_acc
    }

print("✅ Evaluation function ready!")

✅ Evaluation function ready!


In [10]:
# ============================================
# XGBOOST MODEL V1
# ============================================

from xgboost import XGBRegressor

print("="*60)
print("XGBOOST REGRESSOR")
print("="*60)

# Best starting configuration for small datasets
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=2,
    subsample=0.6,
    colsample_bytree=0.6,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0
)

# Train
print("\n🔧 Training XGBoost...")
xgb_model.fit(X_train_scaled, y_train, verbose=False)
print("✅ Training complete!")

# Predictions
y_pred_train = xgb_model.predict(X_train_scaled)
y_pred_val   = xgb_model.predict(X_val_scaled)

# Evaluate
train_metrics = evaluate_model(y_train, y_pred_train, "TRAIN")
val_metrics   = evaluate_model(y_val,   y_pred_val,   "VALIDATION")

# Overfitting check
gap = train_metrics["directional_accuracy"] - val_metrics["directional_accuracy"]
print(f"\n🔍 Overfitting Check:")
print(f"   Train Acc: {train_metrics['directional_accuracy']*100:.2f}%")
print(f"   Val Acc:   {val_metrics['directional_accuracy']*100:.2f}%")
print(f"   Gap:       {gap*100:.2f}%")

if gap > 0.10:
    print("   ⚠️  Overfitting detected (gap > 10%)")
elif gap > 0.05:
    print("   ⚠️  Slight overfitting (gap 5-10%) - acceptable")
else:
    print("   ✅ Good generalization! (gap < 5%)")

XGBOOST REGRESSOR

🔧 Training XGBoost...
✅ Training complete!

📊 TRAIN Performance:
   MAE:                  0.031570  (3.1570%)
   RMSE:                 0.046391  (4.6391%)
   R²:                   0.0152
   Directional Accuracy: 0.5353 (53.53%)

📊 VALIDATION Performance:
   MAE:                  0.018865  (1.8865%)
   RMSE:                 0.028297  (2.8297%)
   R²:                   -0.0025
   Directional Accuracy: 0.4882 (48.82%)

🔍 Overfitting Check:
   Train Acc: 53.53%
   Val Acc:   48.82%
   Gap:       4.72%
   ✅ Good generalization! (gap < 5%)


R² ≈ 0 or negative

This is normal in financial return prediction.

Returns are:

Noisy

Low signal-to-noise ratio

Hard to predict

In finance, even very strong models often have:
R² close to zero.

So this is NOT shocking.

In [11]:
# ============================================
# XGBOOST - MULTIPLE CONFIGURATIONS
# ============================================

from xgboost import XGBRegressor
import numpy as np

print("="*60)
print("XGBOOST - CONFIGURATION SEARCH")
print("="*60)

configs = {
    "Very Conservative": {
        "n_estimators": 50,
        "learning_rate": 0.005,
        "max_depth": 2,
        "subsample": 0.5,
        "colsample_bytree": 0.5,
        "min_child_weight": 10,
        "reg_alpha": 1.0,
        "reg_lambda": 2.0,
        "random_state": 42,
        "verbosity": 0
    },
    "Conservative": {
        "n_estimators": 100,
        "learning_rate": 0.01,
        "max_depth": 2,
        "subsample": 0.6,
        "colsample_bytree": 0.6,
        "min_child_weight": 5,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "random_state": 42,
        "verbosity": 0
    },
    "Moderate": {
        "n_estimators": 150,
        "learning_rate": 0.02,
        "max_depth": 3,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
        "min_child_weight": 3,
        "reg_alpha": 0.05,
        "reg_lambda": 1.0,
        "random_state": 42,
        "verbosity": 0
    }
}

results = {}

for config_name, params in configs.items():
    model = XGBRegressor(**params)
    model.fit(X_train_scaled, y_train, verbose=False)
    
    y_pred_train = model.predict(X_train_scaled)
    y_pred_val   = model.predict(X_val_scaled)
    
    train_metrics = evaluate_model(y_train, y_pred_train, f"{config_name} TRAIN")
    val_metrics   = evaluate_model(y_val,   y_pred_val,   f"{config_name} VAL")
    
    gap = train_metrics["directional_accuracy"] - val_metrics["directional_accuracy"]
    print(f"   Gap: {gap*100:.2f}%\n")
    
    results[config_name] = {
        "model": model,
        "train": train_metrics,
        "val": val_metrics,
        "gap": gap
    }

# Summary table
print("\n" + "="*60)
print("📊 SUMMARY:")
print("="*60)
print(f"\n{'Config':<20} {'Train Acc':>10} {'Val Acc':>10} {'Gap':>8}")
print("-"*52)

best_val_acc  = 0
best_config   = None

for name, result in results.items():
    train_acc = result["train"]["directional_accuracy"]
    val_acc   = result["val"]["directional_accuracy"]
    gap       = result["gap"]
    
    marker = " 🏆" if val_acc > best_val_acc else ""
    print(f"{name:<20} {train_acc*100:>9.2f}% {val_acc*100:>9.2f}% {gap*100:>7.2f}%{marker}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_config  = name

print(f"\n🏆 Best: {best_config} → Val Acc: {best_val_acc*100:.2f}%")
best_xgb_model = results[best_config]["model"]

XGBOOST - CONFIGURATION SEARCH

📊 Very Conservative TRAIN Performance:
   MAE:                  0.031698  (3.1698%)
   RMSE:                 0.046690  (4.6690%)
   R²:                   0.0025
   Directional Accuracy: 0.5269 (52.69%)

📊 Very Conservative VAL Performance:
   MAE:                  0.018877  (1.8877%)
   RMSE:                 0.028278  (2.8278%)
   R²:                   -0.0012
   Directional Accuracy: 0.5008 (50.08%)
   Gap: 2.61%


📊 Conservative TRAIN Performance:
   MAE:                  0.031570  (3.1570%)
   RMSE:                 0.046391  (4.6391%)
   R²:                   0.0152
   Directional Accuracy: 0.5353 (53.53%)

📊 Conservative VAL Performance:
   MAE:                  0.018865  (1.8865%)
   RMSE:                 0.028297  (2.8297%)
   R²:                   -0.0025
   Directional Accuracy: 0.4882 (48.82%)
   Gap: 4.72%


📊 Moderate TRAIN Performance:
   MAE:                  0.030934  (3.0934%)
   RMSE:                 0.045108  (4.5108%)
   R²:            

insights : 
- Slight overfitting only 7% (acceptable range)
- Val Accuracy = 49.32% (slightly below random!)
reason : 
- dataset is too small 
- XGBoost might not be the right model: XGBoost excels at tabular data with clear patterns


#### 4.2 Random Forest

In [12]:
# ============================================
# MODEL 2: RANDOM FOREST
# ============================================

from sklearn.ensemble import RandomForestRegressor

print("="*60)
print("RANDOM FOREST REGRESSOR")
print("="*60)

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=3,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features=0.6,
    random_state=42,
    n_jobs=-1
)

print("\n🔧 Training Random Forest...")
rf_model.fit(X_train_scaled, y_train)
print("✅ Training complete!")

y_pred_train_rf = rf_model.predict(X_train_scaled)
y_pred_val_rf   = rf_model.predict(X_val_scaled)

train_metrics_rf = evaluate_model(y_train, y_pred_train_rf, "TRAIN")
val_metrics_rf   = evaluate_model(y_val,   y_pred_val_rf,   "VALIDATION")

gap_rf = train_metrics_rf["directional_accuracy"] - val_metrics_rf["directional_accuracy"]
print(f"\n🔍 Overfitting Check:")
print(f"   Gap: {gap_rf*100:.2f}%")
if gap_rf > 0.10:
    print("   ⚠️  Overfitting detected!")
elif gap_rf > 0.05:
    print("   ⚠️  Slight overfitting - acceptable")
else:
    print("   ✅ Good generalization!")

RANDOM FOREST REGRESSOR

🔧 Training Random Forest...
✅ Training complete!

📊 TRAIN Performance:
   MAE:                  0.031397  (3.1397%)
   RMSE:                 0.046052  (4.6052%)
   R²:                   0.0295
   Directional Accuracy: 0.5438 (54.38%)

📊 VALIDATION Performance:
   MAE:                  0.018903  (1.8903%)
   RMSE:                 0.028331  (2.8331%)
   R²:                   -0.0049
   Directional Accuracy: 0.4961 (49.61%)

🔍 Overfitting Check:
   Gap: 4.77%
   ✅ Good generalization!


#### 4.3 LightGBM

In [13]:
# ============================================
# MODEL 3: LIGHTGBM
# ============================================

try:
    import lightgbm as lgb
    print("\n✅ LightGBM already installed!")
except ImportError:
    print("Installing LightGBM...")
    import subprocess
    subprocess.run(["pip", "install", "lightgbm"], capture_output=True)
    import lightgbm as lgb

print("\n" + "="*60)
print("LIGHTGBM REGRESSOR")
print("="*60)

lgb_model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=3,
    num_leaves=8,
    subsample=0.6,
    colsample_bytree=0.6,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1
)

print("\n🔧 Training LightGBM...")
lgb_model.fit(X_train_scaled, y_train)
print("✅ Training complete!")

y_pred_train_lgb = lgb_model.predict(X_train_scaled)
y_pred_val_lgb   = lgb_model.predict(X_val_scaled)

train_metrics_lgb = evaluate_model(y_train, y_pred_train_lgb, "TRAIN")
val_metrics_lgb   = evaluate_model(y_val,   y_pred_val_lgb,   "VALIDATION")

gap_lgb = train_metrics_lgb["directional_accuracy"] - val_metrics_lgb["directional_accuracy"]
print(f"\n🔍 Overfitting Check:")
print(f"   Gap: {gap_lgb*100:.2f}%")
if gap_lgb > 0.10:
    print("   ⚠️  Overfitting detected!")
elif gap_lgb > 0.05:
    print("   ⚠️  Slight overfitting - acceptable")
else:
    print("   ✅ Good generalization!")



✅ LightGBM already installed!

LIGHTGBM REGRESSOR

🔧 Training LightGBM...
✅ Training complete!

📊 TRAIN Performance:
   MAE:                  0.031500  (3.1500%)
   RMSE:                 0.046247  (4.6247%)
   R²:                   0.0213
   Directional Accuracy: 0.5385 (53.85%)

📊 VALIDATION Performance:
   MAE:                  0.018900  (1.8900%)
   RMSE:                 0.028324  (2.8324%)
   R²:                   -0.0044
   Directional Accuracy: 0.4905 (49.05%)

🔍 Overfitting Check:
   Gap: 4.80%
   ✅ Good generalization!


Why Traditional ML Struggles:
1. **Sequential blindness:** Tree models treat each row independently
   - Cannot learn "RSI was rising for 3 days → bullish pattern"
   - Miss temporal dependencies between days
2. **Small dataset:** Only 217 samples per coin for training
3. **High noise:** Crypto daily returns have very weak signal
4. **No memory:** Cannot remember what happened last week